# Hito 6 — Modelos de ensamble: Random Forest y XGBoost

**Proyecto 1 de Innovacion Tecnologica — Maestria en Inteligencia Artificial Aplicada (MIAA)**
**Universidad Icesi · Cali, Colombia**
**Responsables del Hito 6:** Carlos Alberto Salamando · Veronica Serna
**Equipo:** Juan Fernando Gutierrez · Miguel Angel Paz · Carlos Alberto Salamando · Veronica Serna

---

## Que hace este notebook y como leerlo

Este notebook documenta **de inicio a fin** la metodologia y los resultados del Hito 6:
entrenamiento y comparacion de Random Forest y XGBoost sobre las 90 caracteristicas
acusticas del Hito 3, con busqueda de hiperparametros dentro de la validacion cruzada,
contraste estadistico contra la linea base del Hito 5 (SVM RBF), calibracion de umbrales
de decision e importancia de caracteristicas.

**Orden metodologico de los pasos:**

1. **Configuracion** — rutas, semilla y carga de artefactos.
2. **Configuracion experimental** — que protocolo se uso y por que.
3. **Resultados de validacion cruzada** — metricas por modelo con IC 95 %.
4. **Detalle por fold** — la tabla completa, sin agregacion.
5. **Contraste estadistico** — Wilcoxon pareado contra SVM RBF.
6. **Sensibilidad al umbral** — por que el umbral fijo 0,5 no basta.
7. **Umbrales out-of-fold** — la correccion del sobreajuste de umbrales.
8. **Importancia de caracteristicas** — que features dominan y que significan.
9. **Conclusiones** — sintesis y decision tomada.

**Importante:** la corrida completa (busqueda de hiperparametros en 5 folds) tarda 2–3 h
incluso en GPU, por lo que este notebook **lee los artefactos ya ejecutados** en
`experiments/results/hito6/` y muestra sus salidas reales. Para reproducir la corrida
desde cero:

```bash
uv run python src/main.py --stage hito6            # con GPU si hay CUDA
uv run python src/main.py --stage hito6 --device cpu   # sin GPU NVIDIA (mas lento, mismos resultados)
```

**La particion `test` no se usa en ningun paso de este notebook.**

## Paso 1 — Configuracion: rutas, imports y carga de artefactos

**Que se hace:** se ubican las rutas del proyecto (el notebook funciona tanto abierto desde
la raiz como desde `notebooks/`) y se cargan los artefactos JSON/CSV que la corrida oficial
del Hito 6 dejo en `experiments/results/hito6/`.

**Por que:** separar la ejecucion pesada (pipeline) de la presentacion (notebook) permite
explorar los resultados en segundos sin repetir horas de computo, y garantiza que las cifras
del notebook son exactamente las de la corrida oficial (mismo archivo, sin copias manuales).

**Que esperar:** la configuracion de la corrida — semilla 42, 5 folds, 2 modelos, busqueda
interna GroupKFold(3) por autor, dispositivo XGBoost usado y las 16 especies.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

# Raiz del proyecto, abierto desde la raiz o desde notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS = PROJECT_ROOT / "experiments" / "results" / "hito6"
summary = json.loads((RESULTS / "cv_summary.json").read_text(encoding="utf-8"))
paired = json.loads((RESULTS / "paired_tests_vs_hito5.json").read_text(encoding="utf-8"))
thresholds = json.loads((RESULTS / "threshold_sensitivity.json").read_text(encoding="utf-8"))
oof = json.loads((RESULTS / "oof_thresholds.json").read_text(encoding="utf-8"))
cv_scores = pd.read_csv(RESULTS / "cv_scores_by_fold.csv")
importance = pd.read_csv(RESULTS / "feature_importance.csv")

cfg = summary["config"]
print(f"Semilla: {cfg['seed']} | folds: {cfg['folds']}")
print(f"Modelos: {cfg['models']} | busqueda interna: {cfg['inner_cv']}")
print(f"Dispositivo XGBoost en la corrida oficial: {cfg['xgboost_device']}")
print(f"Especies ({cfg['n_species']}): {', '.join(cfg['species'])}")
print(f"SMOTE: {cfg['smote']}")

Semilla: 42 | folds: ['fold0', 'fold1', 'fold2', 'fold3', 'fold4']
Modelos: ['random_forest', 'xgboost'] | busqueda interna: GroupKFold(3) por autor, solo dentro del fold de entrenamiento
Dispositivo XGBoost en la corrida oficial: cuda
Especies (16): bugtan, chbant1, colcha1, compau, grekis, gycwor1, laufal1, linwoo1, littin1, orcpar, paltan1, roahaw, rumfly1, soulap1, trsowl, wbwwre1
SMOTE: no aplicado: recall macro Hito 5 = 0.60-0.88 (sin colapso); remuestreo multietiqueta queda como trabajo condicional


## Paso 2 — Diseno experimental (sin codigo: el protocolo ya se ejecuto)

**Que se hizo en la corrida oficial** (modulo `src/training/hito6_ensembles.py`):

1. **Tarea multietiqueta de 16 especies**: una especie vale 1 si aparece en `primary_label`
   o en `secondary`; los scores de clips de una misma grabacion se agregan por **maximo**
   antes de evaluar (la unidad de decision es la grabacion).
2. **CV externa de 5 folds agrupados por autor** (`fold0`–`fold4`, fijados en Hitos 1–2):
   ningun autor ni grabacion aparece en entrenamiento y validacion a la vez → sin fuga.
3. **Escalador reajustado dentro de cada fold** con solo su entrenamiento.
4. **Busqueda de hiperparametros dentro de la CV**: `RandomizedSearchCV` (12 candidatos)
   con `GroupKFold(3)` interno por autor, calculado solo sobre el entrenamiento del fold.
   La validacion externa nunca participa del ajuste.
5. **Umbral fijo 0,5** en la comparacion principal (identico al Hito 5); la calibracion
   de umbrales se estudia aparte (pasos 6 y 7).

**Por que este diseno:** responde a la retroalimentacion del evaluador (intervalos de
confianza y pruebas pareadas, no solo media ± std) y mantiene comparabilidad estricta
con la linea base del Hito 5 (SVM RBF: Macro-F1 0,466 ± 0,019).

## Paso 3 — Resultados de validacion cruzada: resumen por modelo

**Que se hace:** se construye la tabla de metricas por modelo con media e **intervalo de
confianza del 95 %** (t de Student sobre los 5 folds) a partir de `cv_summary.json`.

**Por que:** con solo 5 folds la media sola es enganosa; el IC95 muestra si la diferencia
entre modelos es estable o ruido entre folds.

**Que esperar:** XGBoost por encima de SVM RBF (0,466) en Macro-F1; Random Forest con F1
bajo pese a ROC-AUC alto — el patron que motiva el analisis de umbrales (paso 6).

In [2]:
rows = []
for model, metrics in summary["summary_by_model"].items():
    row = {"modelo": model}
    for metric, stats in metrics.items():
        row[metric] = f"{stats['mean']:.3f} [{stats['ci95_low']:.3f}, {stats['ci95_high']:.3f}]"
    rows.append(row)
table = pd.DataFrame(rows).set_index("modelo")
print("Referencias Hito 5 (CV 5 folds, mismos datos):")
print("  Baseline logistico: Macro-F1 = 0,299 ± 0,007 | ROC-AUC = 0,894")
print("  SVM RBF:            Macro-F1 = 0,466 ± 0,019 | ROC-AUC = 0,908")
print()
print(table.to_string())

Referencias Hito 5 (CV 5 folds, mismos datos):
  Baseline logistico: Macro-F1 = 0,299 ± 0,007 | ROC-AUC = 0,894
  SVM RBF:            Macro-F1 = 0,466 ± 0,019 | ROC-AUC = 0,908

                           accuracy              f1_macro       precision_macro          recall_macro         roc_auc_macro
modelo                                                                                                                     
random_forest  0.112 [0.099, 0.126]  0.200 [0.188, 0.211]  0.841 [0.776, 0.905]  0.121 [0.111, 0.131]  0.888 [0.883, 0.893]
xgboost        0.276 [0.186, 0.366]  0.536 [0.506, 0.567]  0.484 [0.397, 0.572]  0.651 [0.591, 0.712]  0.908 [0.905, 0.912]


## Paso 4 — Detalle completo por fold x modelo

**Que se hace:** se imprime la tabla cruda de `cv_scores_by_fold.csv`, una fila por
(modelo, fold), sin ninguna agregacion.

**Por que:** la trazabilidad importa — cualquier media o IC del paso 3 se puede verificar
a mano desde esta tabla, y permite ver la estabilidad entre folds (p. ej. si un fold
atipico arrastra la media).

**Que esperar:** XGBoost ganando fold a fold; tiempos de ejecucion por fold al final.

In [3]:
pd.set_option("display.max_columns", None)
print(cv_scores.to_string(index=False))

 fold         model  n_train_clips  n_val_clips  n_val_recordings  cv_f1_macro_inner_best  accuracy  f1_macro  precision_macro  recall_macro  roc_auc_macro     cmap  seconds
fold0 random_forest          14458         3435              1158                0.098852  0.094991  0.190600         0.912500      0.112466       0.893686 0.567913    205.6
fold0       xgboost          14458         3435              1158                0.514247  0.203800  0.511386         0.424199      0.681146       0.907833 0.613589   1199.9
fold1 random_forest          14353         3540              1153                0.118314  0.121422  0.204983         0.841540      0.124528       0.889377 0.572074    247.2
fold1       xgboost          14353         3540              1153                0.522150  0.354727  0.567996         0.574225      0.607682       0.909072 0.629288   1216.2
fold2 random_forest          14235         3658              1163                0.105631  0.116079  0.195902         0.785446    

## Paso 5 — Contraste estadistico contra SVM RBF (Wilcoxon pareado)

**Que se hace:** se muestran las diferencias de Macro-F1 fold a fold entre cada ensamble
y la linea base SVM RBF, con la prueba pareada de **Wilcoxon**.

**Por que:** los folds son los mismos para todos los modelos, asi que la comparacion es
pareada; Wilcoxon no asume normalidad (prudente con n = 5).

**Que esperar:** todas las diferencias de XGBoost positivas → p = 0,0625, el **minimo
alcanzable con n = 5 folds**. Se reporta como evidencia de direccion consistente, no
como prueba concluyente (potencia limitada).

In [4]:
for model, res in paired.items():
    diffs = ", ".join(f"{d:+.3f}" for d in res["diff_f1_macro_per_fold"])
    print(f"{model} vs SVM RBF:")
    print(f"  diferencias por fold = [{diffs}]")
    print(f"  media = {res['mean_diff']:+.3f} | Wilcoxon p = {res['wilcoxon_p']}")
    print(f"  nota: {res['note']}")
    print()

random_forest vs SVM RBF:
  diferencias por fold = [-0.253, -0.281, -0.289, -0.261, -0.246]
  media = -0.266 | Wilcoxon p = 0.0625
  nota: n=5 folds: potencia baja; interpretar como evidencia orientativa, no concluyente.

xgboost vs SVM RBF:
  diferencias por fold = [+0.068, +0.082, +0.052, +0.097, +0.055]
  media = +0.071 | Wilcoxon p = 0.0625
  nota: n=5 folds: potencia baja; interpretar como evidencia orientativa, no concluyente.



## Paso 6 — Sensibilidad al umbral de decision

**Que se hace:** se compara el Macro-F1 con el umbral fijo 0,5 contra el umbral por
especie que maximiza F1, elegido **sobre el propio fold de validacion** de cada fold
(protocolo de la primera entrega, sin test).

**Por que:** con prevalencias de ~10 % por especie, el umbral 0,5 es demasiado exigente
para probabilidades conservadoras. Random Forest es el caso extremo: precision 0,79–0,91
con recall 0,11–0,13 — el ranking de sus scores es bueno (ROC-AUC 0,888), la decision
binaria con 0,5 no.

**Que esperar:** saltos grandes de F1 en ambos modelos; esto confirma la calibracion de
umbrales como palanca de desempeno y motiva el paso 7.

In [5]:
rows = []
for model, res in thresholds.items():
    rows.append({
        "modelo": model,
        "F1 umbral 0,5": round(summary["summary_by_model"][model]["f1_macro"]["mean"], 3),
        "F1 umbral ajustado": f"{res['f1_macro_mean']:.3f} ± {res['f1_macro_std']:.3f}",
        "umbral medio": round(sum(f["threshold_mean"] for f in res["per_fold"]) / len(res["per_fold"]), 3),
    })
print(pd.DataFrame(rows).set_index("modelo").to_string())
print()
print(thresholds["random_forest"]["note"])

               F1 umbral 0,5 F1 umbral ajustado  umbral medio
modelo                                                       
random_forest          0.200      0.546 ± 0.014         0.220
xgboost                0.536      0.610 ± 0.009         0.661

Umbral por especie elegido maximizando F1 sobre el propio fold de validación (protocolo de la primera entrega). No usa test.


## Paso 7 — Umbrales out-of-fold (OOF): correccion del sobreajuste de umbrales

**Contexto:** la primera evaluacion final ajusto umbrales solo sobre fold4 y el caso
rumfly1 (umbral 0,30) produjo 111 falsos positivos en test. La causa: ajustar 16 umbrales
sobre un solo fold de 3.597 clips sobreajusta.

**Que se hace aqui:** se muestran los umbrales ajustados con las predicciones
**out-of-fold** de los 5 folds — cada clip de train+val fue validado exactamente una vez
por un modelo que no lo vio en entrenamiento (17.893 clips, 5.794 grabaciones). Se
comparan contra los umbrales fold4. Se reproduce con `uv run python scripts/hito6_oof_thresholds.py`.

**Por que:** ~5x mas datos de validacion por especie → umbrales mas estables, sin tocar test.

**Que esperar:** Macro-F1 de validacion 0,570 → 0,593; los umbrales extremos (como el
0,30 de rumfly1) corrigen hacia valores moderados.

In [6]:
print(oof["nota"])
print(f"Macro-F1 en validacion con umbrales fold4: {oof['macro_f1_val_umbrales_fold4']:.3f}")
print(f"Macro-F1 en validacion con umbrales OOF:   {oof['macro_f1_val_umbrales_oof']:.3f}")
print()
rows = []
for sp in cfg["species"]:
    t4, to = oof["thresholds_fold4"][sp], oof["thresholds_oof"][sp]
    marca = "  <-- cambio > 0,10" if abs(to - t4) > 0.10 else ""
    rows.append({"especie": sp, "umbral_fold4": t4, "umbral_oof": to, "cambio": f"{to - t4:+.2f}{marca}"})
print(pd.DataFrame(rows).set_index("especie").to_string())

Umbrales ajustados con predicciones out-of-fold de los 5 folds (17.893 clips; sin test).
Macro-F1 en validacion con umbrales fold4: 0.570
Macro-F1 en validacion con umbrales OOF:   0.593

         umbral_fold4  umbral_oof                    cambio
especie                                                    
bugtan           0.49        0.56                     +0.07
chbant1          0.54        0.72  +0.18  <-- cambio > 0,10
colcha1          0.62        0.64                     +0.02
compau           0.74        0.77                     +0.03
grekis           0.64        0.77  +0.13  <-- cambio > 0,10
gycwor1          0.61        0.64                     +0.03
laufal1          0.56        0.64                     +0.08
linwoo1          0.55        0.52                     -0.03
littin1          0.69        0.72                     +0.03
orcpar           0.63        0.71                     +0.08
paltan1          0.51        0.51                     +0.00
roahaw           0.56        0.7

## Paso 8 — Importancia de caracteristicas

**Que se hace:** se promedia la importancia de cada una de las 90 features entre los 5
folds (top 15 por modelo) y se muestran las figuras generadas por la corrida oficial.

**Por que:** la importancia responde "en que se fija el modelo" — valida que las features
dominantes tengan sentido bioacustico y alimenta la discusion del informe final.

**Que esperar:** MFCC de orden bajo/medio (medias y desviaciones) y ZCR dominando;
Random Forest concentra la importancia (max. ~6,5 %) y XGBoost la reparte (max. ~2 %).

In [7]:
for model in ("random_forest", "xgboost"):
    agg = (
        importance[importance["model"] == model]
        .groupby("feature")["importance"]
        .agg(importancia_media="mean", desviacion="std")
        .nlargest(15, "importancia_media")
        .reset_index()
    )
    print(f"--- {model} (top 15 de 90 features) ---")
    print(agg.to_string(index=False))
    print()

--- random_forest (top 15 de 90 features) ---
               feature  importancia_media  desviacion
     mfcc_06_base_mean           0.064606    0.026248
     mfcc_07_base_mean           0.063873    0.026763
     mfcc_02_base_mean           0.063479    0.012606
spectral_skewness_mean           0.038710    0.017338
     mfcc_12_base_mean           0.031562    0.005805
     mfcc_11_base_mean           0.027062    0.009127
     mfcc_05_base_mean           0.024445    0.006279
     mfcc_08_base_mean           0.019866    0.005575
     mfcc_04_base_mean           0.019829    0.002100
      mfcc_06_base_std           0.019815    0.010870
     mfcc_03_base_mean           0.018400    0.003277
     mfcc_12_delta_std           0.017914    0.008110
    mfcc_12_delta2_std           0.017362    0.007971
     mfcc_01_base_mean           0.016858    0.004650
    mfcc_04_delta2_std           0.016315    0.006114

--- xgboost (top 15 de 90 features) ---
           feature  importancia_media  desviacion

In [8]:
from IPython.display import Image, display
for png in sorted((RESULTS / "figures").glob("feature_importance_*.png")):
    print(png.name)
    display(Image(filename=str(png)))

feature_importance_random_forest.png
<IPython.core.display.Image object>
feature_importance_xgboost.png
<IPython.core.display.Image object>


## Paso 9 — Conclusiones

1. **XGBoost es el mejor modelo del proyecto**: Macro-F1 = 0,536 [IC95 0,506, 0,567] y
   ROC-AUC = 0,908; supera a SVM RBF (0,466) en los 5/5 folds (Wilcoxon ΔF1 = +0,071,
   p = 0,0625 — minimo alcanzable con n = 5). Sus hiperparametros convergen a un modelo
   conservador: learning_rate 0,05, max_depth 4, 300 arboles, subsample 0,7.
2. **El F1 bajo de Random Forest con umbral 0,5 es un artefacto del umbral**, no del
   ranking (ROC-AUC 0,888): con umbral por especie pasa de 0,200 a 0,547. XGBoost sube
   de 0,536 a 0,610 con el mismo ajuste.
3. **Los umbrales tambien se sobreajustan**: el ajuste out-of-fold (paso 7) mejora la
   estimacion de validacion de 0,570 a 0,593 y corrige umbrales extremos. Esta decision
   llevo la evaluacion final en test de Macro-F1 0,554 a **0,592** (ROC-AUC 0,915), con
   validacion ≈ test: sin brecha de generalizacion.
4. **Las features dominantes son bioacusticamente razonables**: MFCC de orden bajo/medio
   (timbre espectral), su variabilidad temporal (delta std) y ZCR.
5. **Siguiente paso documentado**: evaluacion final ya ejecutada
   (`uv run python src/main.py --stage evaluacion_final`); resultados y matrices de
   confusion por especie en `experiments/results/evaluacion_final/` y en el informe
   final (`docs/informe_final.md`).

---

*Reproducibilidad: este notebook se regenera con salidas reales mediante
`uv run python src/training/preprocessing/build_hito6_notebook.py`.*